# Stack Overflow Annual Developer Survey 2025 — Exploratory Data Analysis (EDA)

**Project Name:** Poll Results Visualizer  
**Project Author:** Akash Das  
**Milestone:** Phase 4 of 8 — Exploratory Data Analysis (EDA)  
**Primary Dataset Input:** `data/processed/survey_results_cleaned.parquet` (49,191 rows × 176 columns)  
**Survey Fieldwork:** May 29, 2025 – June 23, 2025  
**Data Rights & License:** Open Database License (ODbL) v1.0, © Stack Exchange, Inc.  

---


## Section 1: Business Context & Analytical Scope

### 1.1 Context & Dataset Provenance
The **Stack Overflow Annual Developer Survey 2025** is the largest public dataset on the global software development profession. Fielded between **May 29, 2025, and June 23, 2025**, the survey captures responses from **49,191 total respondents across 177 countries**.

### 1.2 Target Populations & Denominator Taxonomy
1. **Raw Extract Records ($N_{\text{raw}} = 49,191$):** Total respondent rows present in the canonical public release.
2. **Official Web Published Base ($N_{\text{official}} = 49,009$):** Qualified respondents evaluated on Stack Overflow's online published report after internal screening (e.g. completion speed < 3 minutes, consent criteria). The 182-row difference reflects internal qualification tags omitted from the public microdata.
3. **Question-Specific Answered Count ($N_{\text{answered}}$):** Because question participation is voluntary, analytical denominators must be calculated against answered responses rather than total rows. All findings herein report the exact $N_{\text{answered}}$ denominator.

### 1.3 Methodological Guardrails for Phase 4 EDA
- **Exploratory, Non-Causal:** Patterns describe the observed respondent sample; no causal inferences or unwarranted generalizations are claimed.
- **Outlier Preservation:** Outliers are analyzed through robust non-parametric metrics (medians, interquartile ranges) rather than arbitrary deletion.
- **Data Authenticity:** Original categories are preserved without arbitrary grouping or synthetic imputation.


---
## Section 2: Environment Setup & High-Performance Data Ingestion

We load the cleaned survey dataset (`survey_results_cleaned.parquet`) generated by the Phase 3 cleaning pipeline. Columnar Parquet storage provides vectorized I/O, loading the 176-column dataset in sub-second time.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Configure project root in sys.path robustly
current_dir = Path.cwd()
if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.stackoverflow_2025 import (
    load_cleaned_survey_data,
    load_survey_schema,
    get_question_text,
    get_survey_kpis,
    analyze_single_select,
    analyze_multi_select,
    cross_tabulate_single,
    get_compensation_summary
)

# Output directory for standalone chart exports
OUTPUT_CHARTS_DIR = PROJECT_ROOT / "outputs" / "charts"
OUTPUT_CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Load cleaned survey microdata and schema dictionary
df = load_cleaned_survey_data(PROJECT_ROOT / "data" / "processed" / "survey_results_cleaned.parquet")
schema_df = load_survey_schema()

print(f"Loaded Cleaned Survey Dataset: {df.shape[0]:,} rows by {df.shape[1]:,} columns.")
print(f"Loaded Schema Question Dictionary: {schema_df.shape[0]:,} questions/sub-questions.")


In [ ]:
# Display High-Level Survey Telemetry KPIs
kpis = get_survey_kpis(df)
print("=== OFFICIAL 2025 SURVEY TELEMETRY KPIs ===")
for k, v in kpis.items():
    print(f"  {k}: {v}")


---
## Section 3: Dataset Profiling & Numerical Variable Overview

### 3.1 Numerical Variables Profiling
The cleaned dataset contains three primary continuous/discrete numerical analytical variables:
- `YearsCode_num`: Total coding experience in years (including education; Mean = 16.57, Median = 14.0).
- `WorkExp_num`: Professional coding work experience in years (Mean = 13.37, Median = 10.0).
- `ConvertedCompYearly`: Official annual USD compensation converted by Stack Overflow at June 25, 2025 exchange rates (Median = $75,320 USD).

*Phase 3 Reconciliation Note:* The means of 16.57 for `YearsCode_num` and 13.37 for `WorkExp_num` reflect the authoritative dataset values and reconcile with `reports/data_cleaning_report.md` (lines 111-112); previously cited numbers (15.65 and 11.83) in a review briefing were typographical errata.


In [ ]:
# Compute summary statistics across all numerical analytical variables
num_cols = ["YearsCode_num", "WorkExp_num", "ConvertedCompYearly"]
profile_rows = []

for col in num_cols:
    s = df[col].dropna()
    profile_rows.append({
        "Variable": col,
        "Answered (N)": f"{len(s):,}",
        "Missing (N)": f"{df[col].isna().sum():,}",
        "Answered (%)": f"{len(s) / len(df) * 100:.2f}%",
        "Min": f"{s.min():,.0f}",
        "25th Pct": f"{s.quantile(0.25):,.0f}",
        "Median": f"{s.median():,.0f}",
        "75th Pct": f"{s.quantile(0.75):,.0f}",
        "Mean": f"{s.mean():,.2f}",
        "Max": f"{s.max():,.0f}",
        "Std Dev": f"{s.std():,.2f}",
    })

profile_df = pd.DataFrame(profile_rows)
print("=== NUMERICAL VARIABLE PROFILING SUMMARY ===")
print(profile_df.to_string(index=False))


---
## Section 4: Demographic & Experience Exploration

We explore respondent age cohorts, geographic concentrations, educational attainment, developer roles, and years of experience.


In [ ]:
# 4.1 Age Cohort Distribution (N = 49,191, 100% complete)
age_df = analyze_single_select(df, "Age")
print("=== AGE COHORT DISTRIBUTION ===")
print(age_df.to_string(index=False))

fig_age = px.bar(
    age_df,
    x="category",
    y="response_count",
    text=age_df["percentage"].apply(lambda v: f"{v:.1f}%"),
    title="Respondent Age Distribution (N = 49,191, 100% Complete)",
    labels={"response_count": "Respondents (N)", "category": "Age Cohort"},
    color="response_count",
    color_continuous_scale="Blues"
)
fig_age.update_layout(template="plotly_white", showlegend=False, xaxis_tickangle=-25)
fig_age.write_html(OUTPUT_CHARTS_DIR / "01_age_distribution.html")
fig_age.show()


In [ ]:
# 4.2 Geographic Concentration (Top 10 Countries, N_answered = 35,408)
country_df = analyze_single_select(df, "Country", top_n=10)
print("=== TOP 10 RESPONDENT COUNTRIES ===")
print(country_df.to_string(index=False))

fig_country = px.bar(
    country_df,
    x="percentage",
    y="category",
    orientation="h",
    text=country_df["percentage"].apply(lambda v: f"{v:.1f}%"),
    title=f"Top 10 Respondent Countries (N_answered = {df['Country'].notna().sum():,}, Missing = {df['Country'].isna().sum():,})",
    labels={"percentage": "Share of Answered Respondents (%)", "category": "Country"},
    color="percentage",
    color_continuous_scale="Greens"
)
fig_country.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white", showlegend=False)
fig_country.write_html(OUTPUT_CHARTS_DIR / "02_top_countries.html")
fig_country.show()


In [ ]:
# 4.3 Educational Attainment & Primary Developer Roles
ed_df = analyze_single_select(df, "EdLevel")
print("=== EDUCATIONAL ATTAINMENT (N_answered = 48,149) ===")
print(ed_df.to_string(index=False))

devtype_df = analyze_single_select(df, "DevType", top_n=15)
print()
print("=== TOP 15 DEVELOPER ROLES (N_answered = 43,680) ===")
print(devtype_df.to_string(index=False))

fig_dev = px.bar(
    devtype_df,
    x="percentage",
    y="category",
    orientation="h",
    text=devtype_df["percentage"].apply(lambda v: f"{v:.1f}%"),
    title="Top 15 Developer Roles (N_answered = 43,680, Single-Select)",
    labels={"percentage": "Share of Answered Respondents (%)", "category": "Developer Role"},
    color="percentage",
    color_continuous_scale="Oranges"
)
fig_dev.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white", showlegend=False)
fig_dev.write_html(OUTPUT_CHARTS_DIR / "03_developer_roles.html")
fig_dev.show()


In [ ]:
# 4.4 Coding vs. Professional Work Experience Distributions
fig_exp = go.Figure()
fig_exp.add_trace(go.Histogram(
    x=df["YearsCode_num"].dropna(),
    name=f"Total Coding Experience (Median: {df['YearsCode_num'].median():.0f} yrs, Mean: {df['YearsCode_num'].mean():.1f} yrs)",
    nbinsx=40,
    opacity=0.7,
    marker_color="#1f77b4"
))
fig_exp.add_trace(go.Histogram(
    x=df["WorkExp_num"].dropna(),
    name=f"Professional Work Experience (Median: {df['WorkExp_num'].median():.0f} yrs, Mean: {df['WorkExp_num'].mean():.1f} yrs)",
    nbinsx=40,
    opacity=0.7,
    marker_color="#2ca02c"
))
fig_exp.update_layout(
    barmode="overlay",
    title=f"Experience Distributions in Years (YearsCode N={df['YearsCode_num'].notna().sum():,}; WorkExp N={df['WorkExp_num'].notna().sum():,})",
    xaxis_title="Years of Experience",
    yaxis_title="Respondent Count",
    template="plotly_white"
)
fig_exp.write_html(OUTPUT_CHARTS_DIR / "04_experience_distributions.html")
fig_exp.show()


---
## Section 5: Technology Landscape & Multi-Select Analysis

### 5.1 Dual-Percentage Denominator Methodology
In multi-select questions (semicolon-delimited lists), a respondent may select multiple options. We evaluate two distinct percentages:
- **Respondent-Level Percentage (% of Answered Respondents):** $\frac{\text{Selections for Option}}{N_{\text{answered}}} \times 100$. Answers: *"What percentage of participating developers use this technology?"*
- **Selection-Level Percentage (% of Total Selections):** $\frac{\text{Selections for Option}}{\text{Total Selections Made across All Options}} \times 100$. Answers: *"What share of overall technology mentions does this tool represent?"*


In [ ]:
# 5.1 Programming Languages: Have Worked With vs. Want to Work With
lang_have = analyze_multi_select(df, "LanguageHaveWorkedWith", top_n=15)
lang_want = analyze_multi_select(df, "LanguageWantToWorkWith", top_n=15)

print("=== TOP 10 PROGRAMMING LANGUAGES (WORKED WITH, N_answered = 31,671) ===")
print(lang_have.head(10).to_string(index=False))

# Current-versus-desired respondent-share gap (percentage points)
paired_lang_n = int((df["LanguageHaveWorkedWith"].notna() & df["LanguageWantToWorkWith"].notna()).sum())
print(f"LanguageHaveWorkedWith N_answered = {df['LanguageHaveWorkedWith'].notna().sum():,}")
print(f"LanguageWantToWorkWith N_answered = {df['LanguageWantToWorkWith'].notna().sum():,}")
print(f"Paired matched sample size (answering both) = {paired_lang_n:,}")

top_langs = lang_have.head(12)["option"].tolist()
comp_lang = []
for lang in top_langs:
    have_pct = lang_have.loc[lang_have["option"] == lang, "pct_of_respondents"].values[0]
    want_match = lang_want.loc[lang_want["option"] == lang, "pct_of_respondents"]
    want_pct = want_match.values[0] if len(want_match) > 0 else 0.0
    comp_lang.append({
        "Language": lang,
        "Worked_With (%)": have_pct,
        "Want_To_Work (%)": want_pct,
        "Share Gap (pp)": round(want_pct - have_pct, 2)
    })

# Add Rust if not in top 12
if "Rust" not in top_langs:
    rust_have = lang_have.loc[lang_have["option"] == "Rust", "pct_of_respondents"].values[0]
    rust_want = lang_want.loc[lang_want["option"] == "Rust", "pct_of_respondents"].values[0]
    comp_lang.append({
        "Language": "Rust",
        "Worked_With (%)": rust_have,
        "Want_To_Work (%)": rust_want,
        "Share Gap (pp)": round(rust_want - rust_have, 2)
    })

comp_lang_df = pd.DataFrame(comp_lang).sort_values("Worked_With (%)", ascending=False)
print()
print("=== PROGRAMMING LANGUAGE: CURRENT VS. DESIRED RESPONDENT-SHARE GAP (pp) ===")
print(comp_lang_df.to_string(index=False))

fig_lang = go.Figure()
fig_lang.add_trace(go.Bar(
    x=comp_lang_df["Language"],
    y=comp_lang_df["Worked_With (%)"],
    name=f"Worked With (N={df['LanguageHaveWorkedWith'].notna().sum():,})",
    marker_color="#1f77b4"
))
fig_lang.add_trace(go.Bar(
    x=comp_lang_df["Language"],
    y=comp_lang_df["Want_To_Work (%)"],
    name=f"Want to Work With (N={df['LanguageWantToWorkWith'].notna().sum():,})",
    marker_color="#ff7f0e"
))
fig_lang.update_layout(
    barmode="group",
    title=f"Top Programming Languages: Current Use vs. Future Stated Interest (% of Answered; Paired N = {paired_lang_n:,})",
    xaxis_title="Language",
    yaxis_title="Respondent Adoption / Stated Interest (% of Answered)",
    template="plotly_white"
)
fig_lang.write_html(OUTPUT_CHARTS_DIR / "05_top_languages_have_vs_want.html")
fig_lang.show()


In [ ]:
# 5.2 Databases & Cloud / Infrastructure Platforms
db_have = analyze_multi_select(df, "DatabaseHaveWorkedWith", top_n=10)
plat_have = analyze_multi_select(df, "PlatformHaveWorkedWith", top_n=10)

print("=== TOP 10 DATABASES (N_answered = 25,553) ===")
print(db_have.to_string(index=False))

fig_db = px.bar(
    db_have,
    x="pct_of_respondents",
    y="option",
    orientation="h",
    text=db_have["pct_of_respondents"].apply(lambda v: f"{v:.1f}%"),
    title=f"Top 10 Databases Worked With (N_answered = {df['DatabaseHaveWorkedWith'].notna().sum():,})",
    labels={"pct_of_respondents": "Share of Answered Respondents (%)", "option": "Database"},
    color="pct_of_respondents",
    color_continuous_scale="Purples"
)
fig_db.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white", showlegend=False)
fig_db.write_html(OUTPUT_CHARTS_DIR / "06_top_databases.html")
fig_db.show()

fig_plat = px.bar(
    plat_have,
    x="pct_of_respondents",
    y="option",
    orientation="h",
    text=plat_have["pct_of_respondents"].apply(lambda v: f"{v:.1f}%"),
    title=f"Top 10 Cloud & Infrastructure Platforms (N_answered = {df['PlatformHaveWorkedWith'].notna().sum():,})",
    labels={"pct_of_respondents": "Share of Answered Respondents (%)", "option": "Platform"},
    color="pct_of_respondents",
    color_continuous_scale="Teal"
)
fig_plat.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white", showlegend=False)
fig_plat.write_html(OUTPUT_CHARTS_DIR / "07_top_cloud_platforms.html")
fig_plat.show()


---
## Section 6: AI Tool Adoption, Trust & Sentiment

We explore developer engagement with AI tools, the trust-capability paradox, and the emergence of autonomous AI agents.


In [ ]:
# 6.1 Daily/Weekly AI Developer Tool Usage (AISelect, N_answered = 33,720)
ai_usage = analyze_single_select(df, "AISelect")
print("=== AI TOOL ADOPTION FREQUENCY (AISelect) ===")
print(ai_usage.to_string(index=False))

fig_ai_usage = px.pie(
    ai_usage,
    names="category",
    values="response_count",
    title=f"Current AI Tool Usage in Development Process (AISelect, N_answered = {df['AISelect'].notna().sum():,})",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig_ai_usage.write_html(OUTPUT_CHARTS_DIR / "08_ai_tool_usage.html")
fig_ai_usage.show()


In [ ]:
# 6.2 The Trust-Capability Paradox: Trust in Accuracy (AIAcc) vs. Complex Tasks (AIComplex)
ai_acc = analyze_single_select(df, "AIAcc")
ai_complex = analyze_single_select(df, "AIComplex")

active_mask = df["AISelect"].isin(["Yes, I use AI tools daily", "Yes, I use AI tools weekly", "Yes, I use AI tools monthly or infrequently"])
acc_active_n = int((active_mask & df["AIAcc"].notna()).sum())
complex_active_n = int((active_mask & df["AIComplex"].notna()).sum())

print(f"=== TRUST IN ACCURACY OF AI TOOL OUTPUTS (AIAcc, Total N_answered = {df['AIAcc'].notna().sum():,}; Active AI Users N = {acc_active_n:,}) ===")
print(ai_acc.to_string(index=False))
print()
print(f"=== PERCEIVED ABILITY OF AI TO HANDLE COMPLEX TASKS (AIComplex, Total N_answered = {df['AIComplex'].notna().sum():,}; Active AI Users N = {complex_active_n:,}) ===")
print(ai_complex.to_string(index=False))

fig_ai_acc = px.bar(
    ai_acc,
    x="category",
    y="percentage",
    text=ai_acc["percentage"].apply(lambda v: f"{v:.1f}%"),
    title=f"Trust in Accuracy of Output from AI Tools (AIAcc, Total N = {df['AIAcc'].notna().sum():,}; Active Users N = {acc_active_n:,})",
    labels={"percentage": "Share of Answered Respondents (%)", "category": "Trust Level"},
    color="category",
    color_discrete_sequence=["#2ca02c", "#1f77b4", "#ff7f0e", "#d62728"]
)
fig_ai_acc.update_layout(template="plotly_white", showlegend=False, xaxis_tickangle=-15)
fig_ai_acc.write_html(OUTPUT_CHARTS_DIR / "09_ai_trust_accuracy.html")
fig_ai_acc.show()


In [ ]:
# 6.3 Autonomous AI Agent Adoption (AIAgents, Total N_answered = 31,919)
ai_agents = analyze_single_select(df, "AIAgents")
agents_active_n = int((active_mask & df["AIAgents"].notna()).sum())
print(f"=== AUTONOMOUS AI AGENT ADOPTION (AIAgents, Total N_answered = {df['AIAgents'].notna().sum():,}; Active AI Users N = {agents_active_n:,}) ===")
print(ai_agents.to_string(index=False))

fig_agents = px.bar(
    ai_agents,
    x="percentage",
    y="category",
    orientation="h",
    text=ai_agents["percentage"].apply(lambda v: f"{v:.1f}%"),
    title=f"Adoption of Autonomous AI Agents in Workflows (AIAgents, Total N = {df['AIAgents'].notna().sum():,}; Active Users N = {agents_active_n:,})",
    labels={"percentage": "Share of Answered Respondents (%)", "category": "Agent Adoption"},
    color="percentage",
    color_continuous_scale="Purples"
)
fig_agents.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white", showlegend=False)
fig_agents.write_html(OUTPUT_CHARTS_DIR / "10_ai_agents_adoption.html")
fig_agents.show()


---
## Section 7: Work Arrangements & Annual Compensation Exploration

### 7.1 Compensation Methodology & Taxonomies
- **`ConvertedCompYearly`:** Pre-converted annual USD compensation provided by Stack Overflow based on exchange rates as of June 25, 2025 ($N_{\text{answered}} = 23,947$).
- **`Comp_status` Taxonomy:** 4 mutually exclusive states: `reported_usd` (23,947; 48.68%), `omitted` (24,325; 49.45%), `unconverted_local` (919; 1.87%), and `non_positive_usd` (0; 0.00%).
- **Population Reconciliation:** The project analyzes all 23,947 raw-extract salary observations. The official published benchmark uses 23,928 qualified salary respondents. The public microdata does not expose sufficient qualification-screening information to identify individual excluded records; this variance is therefore a population-count reconciliation limitation between the raw public extract and the official published qualified respondent population, rather than a confirmed record-level screening result.
- **Caveats:** All figures represent self-reported, unverified compensation. No purchasing power parity (PPP) adjustments are applied.


In [ ]:
# 7.1 Remote vs. Hybrid vs. In-Person Work Arrangements (RemoteWork, N_answered = 33,780)
remote_df = analyze_single_select(df, "RemoteWork")
print("=== WORK ARRANGEMENTS DISTRIBUTION (N_answered = 33,780) ===")
print(remote_df.to_string(index=False))


In [ ]:
# 7.2 Annual Compensation by Country (ConvertedCompYearly in USD)
top_comp_countries = [
    'United States of America',
    'Germany',
    'United Kingdom of Great Britain and Northern Ireland',
    'Canada',
    'France',
    'India',
    'Australia',
    'Netherlands',
    'Switzerland'
]

comp_country = get_compensation_summary(
    df[df['Country'].isin(top_comp_countries)],
    group_by='Country',
    trim_outliers=False
)

print("=== CONVERTED ANNUAL USD COMPENSATION BY TOP COUNTRIES (N_answered >= 100) ===")
print(comp_country[['group', 'n_answered', 'median_usd', 'p25_usd', 'p75_usd', 'iqr_usd']].to_string(index=False))

fig_comp_country = px.bar(
    comp_country,
    x="median_usd",
    y="group",
    orientation="h",
    text=comp_country["median_usd"].apply(lambda v: f"${v:,.0f}"),
    title="Median Annual Compensation (Converted USD) by Country (Countries with N >= 100 responses)",
    labels={"median_usd": "Median Annual Compensation (USD)", "group": "Country"},
    color="median_usd",
    color_continuous_scale="Greens"
)
fig_comp_country.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white", showlegend=False)
fig_comp_country.write_html(OUTPUT_CHARTS_DIR / "11_compensation_by_country.html")
fig_comp_country.show()


In [ ]:
# 7.3 Compensation by Professional Experience Bracket (Mutually Exclusive & Exhaustive)
valid_comp = df[df["Comp_valid_usd"]].copy()
bins_exp = [0, 2, 6, 11, 16, 20, np.inf]
labels_exp = ["< 2 years", "2 to under 6 years", "6 to under 11 years", "11 to under 16 years", "16 to under 20 years", "20 years and above"]
valid_comp["Exp_Bracket"] = pd.cut(
    valid_comp["WorkExp_num"],
    bins=bins_exp,
    labels=labels_exp,
    right=False
)

exp_comp = (
    valid_comp.groupby("Exp_Bracket", observed=False)["ConvertedCompYearly"]
    .agg(["count", "median", lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)])
    .rename(columns={"<lambda_0>": "p25_usd", "<lambda_1>": "p75_usd"})
    .reset_index()
)
print("=== COMPENSATION BY PROFESSIONAL WORK EXPERIENCE BRACKET ===")
print(exp_comp.to_string(index=False))

fig_comp_exp = px.bar(
    exp_comp,
    x="Exp_Bracket",
    y="median",
    text=exp_comp["median"].apply(lambda v: f"${v:,.0f}"),
    labels={"median": "Median Annual USD ($)", "Exp_Bracket": "Professional Work Experience"},
    title=f"Median Annual Compensation by Experience Bracket (N_answered = {valid_comp['Exp_Bracket'].notna().sum():,}; Missing = {valid_comp['Exp_Bracket'].isna().sum():,})",
    color="median",
    color_continuous_scale="Blues"
)
fig_comp_exp.update_layout(template="plotly_white", showlegend=False)
fig_comp_exp.write_html(OUTPUT_CHARTS_DIR / "12_compensation_by_experience.html")
fig_comp_exp.show()


In [ ]:
# 7.4 Compensation by Developer Role (Roles with N >= 50 salary responses)
role_comp = (
    valid_comp.groupby("DevType")["ConvertedCompYearly"]
    .agg(["count", "median", lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)])
    .rename(columns={"<lambda_0>": "p25_usd", "<lambda_1>": "p75_usd"})
    .query("count >= 50")
    .sort_values("median", ascending=False)
    .head(12)
    .reset_index()
)
print("=== TOP DEVELOPER ROLES BY MEDIAN ANNUAL USD COMPENSATION ===")
print(role_comp.to_string(index=False))

fig_comp_role = px.bar(
    role_comp,
    x="median",
    y="DevType",
    orientation="h",
    text=role_comp["median"].apply(lambda v: f"${v:,.0f}"),
    labels={"median": "Median Annual USD ($)", "DevType": "Developer Role"},
    title="Top Developer Roles by Median Annual Compensation (Converted USD, N >= 50 responses)",
    color="median",
    color_continuous_scale="Oranges"
)
fig_comp_role.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white", showlegend=False)
fig_comp_role.write_html(OUTPUT_CHARTS_DIR / "13_compensation_by_role.html")
fig_comp_role.show()


---
## Section 8: Inter-Variable Associations & Cross-Tabulations

We evaluate descriptive relationships across demographic segments, work situations, and technology behaviors.


In [ ]:
# 8.1 Age Cohort vs. AI Tool Adoption (AISelect x Age)
age_ai = cross_tabulate_single(df, "AISelect", "Age")
print("=== AI TOOL USAGE BY AGE COHORT (Head) ===")
print(age_ai.head(10).to_string(index=False))

fig_age_ai = px.bar(
    age_ai,
    x="Age",
    y="percentage",
    color="category",
    barmode="stack",
    title="AI Tool Adoption by Age Cohort (% within Age Cohort)",
    labels={"percentage": "Share within Age Cohort (%)", "Age": "Age Cohort"}
)
fig_age_ai.update_layout(template="plotly_white", xaxis_tickangle=-25)
fig_age_ai.show()


In [ ]:
# 8.2 Empirical Cross-Tabulation: RemoteWork Missingness by Employment Category
emp = df["Employment"].fillna("Missing Employment")
ct_emp_rw = pd.crosstab(emp, df["RemoteWork"].isna(), rownames=["Employment"], colnames=["RemoteWork_Missing"])
ct_emp_rw["Total"] = ct_emp_rw[False] + ct_emp_rw[True]
ct_emp_rw["Pct_Missing"] = (ct_emp_rw[True] / ct_emp_rw["Total"] * 100).round(2)
ct_emp_rw = ct_emp_rw.rename(columns={False: "Answered_RemoteWork", True: "Missing_RemoteWork"}).sort_values("Total", ascending=False)

print("=== EMPIRICAL CROSS-TABULATION: REMOTEWORK MISSINGNESS BY EMPLOYMENT ===")
print(ct_emp_rw.to_string())


---
## Section 9: Missingness Patterns & Survey Quality Exploration

We profile missingness rates across key sections of the survey instrument to uncover skip logic and survey attrition.


In [ ]:
# 9.1 Survey Section Missingness Audit
key_sections = {
    "Demographics": ["Age", "EdLevel", "Country", "DevType", "Employment", "RemoteWork"],
    "Experience": ["YearsCode", "WorkExp", "YearsCode_num", "WorkExp_num"],
    "Technology Use": ["LanguageHaveWorkedWith", "DatabaseHaveWorkedWith", "PlatformHaveWorkedWith", "WebframeHaveWorkedWith"],
    "AI Module": ["AISelect", "AISent", "AIAcc", "AIComplex", "AIAgents"],
    "Compensation": ["Currency", "CompTotal", "ConvertedCompYearly", "Comp_status"]
}

missing_rows = []
for section, cols in key_sections.items():
    for col in cols:
        missing_rows.append({
            "Section": section,
            "Survey Field": col,
            "Missing Count (N)": df[col].isna().sum(),
            "Missing Rate (%)": round(df[col].isna().sum() / len(df) * 100, 2),
            "Answered Count (N)": df[col].notna().sum(),
            "Answered Rate (%)": round(df[col].notna().sum() / len(df) * 100, 2),
        })

missing_df = pd.DataFrame(missing_rows)
print("=== MISSINGNESS AUDIT ACROSS KEY SURVEY MODULES (Head) ===")
print(missing_df.head(15).to_string(index=False))

fig_missing = px.bar(
    missing_df,
    x="Missing Rate (%)",
    y="Survey Field",
    color="Section",
    orientation="h",
    title="Missingness Rate Across Key Survey Variables by Section (Total N = 49,191)",
    labels={"Missing Rate (%)": "Missing Rate (%)", "Survey Field": "Survey Field"}
)
fig_missing.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white", height=700)
fig_missing.write_html(OUTPUT_CHARTS_DIR / "14_missingness_by_survey_section.html")
fig_missing.show()


---
## Section 10: Summary of Exploratory Observations & Phase 5 Questions

### 10.1 Key Exploratory Takeaways
1. **Pervasive AI Adoption Coupled with Deep Skepticism:** While 78.5% of answered developers actively use AI tools (47.1% daily), only 3.15% of all answered respondents (and 3.82% of active users) express high trust in their accuracy, and 47.6% rate them as poor at handling complex engineering tasks.
2. **Rapid Uptake of Autonomous Agents:** Over 30.9% of answered developers report using autonomous AI agents (14.1% daily), demonstrating that agentic engineering has transitioned from research to commercial adoption.
3. **Rust Stated Interest vs. Current Use:** Although JavaScript, HTML/CSS, SQL, and Python command current usage dominance, Rust exhibits the highest positive current-versus-desired respondent-share gap (+19.3 percentage points; paired N = 27,079).
4. **PostgreSQL as Primary Relational Standard:** PostgreSQL leads all databases at 56.9% adoption among answered respondents.
5. **Geographic Compensation Disparities:** Median converted annual USD compensation ranges from $150,000 USD in the United States ($N=5,243$) to $17,029 USD in India ($N=1,096$), matching Phase 2 benchmarks exactly and highlighting severe geographic divergence in market rates.
6. **Non-Random Survey Missingness:** Survey attrition increases systematically across survey modules, while specific items (e.g. `RemoteWork`, `AIAcc`) exhibit non-random structural skip patterns.

### 10.2 Recommended Questions for Phase 5 (Analysis)
- **Question 1:** What individual developer attributes (experience, role, organization size) are most strongly associated with trust in AI output accuracy?
- **Question 2:** After controlling for years of experience and developer role, what is the residual compensation premium associated with specialized tech stacks (e.g., Rust, Go, Cloud infrastructure)?
- **Question 3:** Does autonomous AI agent adoption vary significantly between remote, hybrid, and in-person developers?
- **Question 4:** How does the distribution of developer roles differ between emerging developer markets (e.g., India) and established software hubs (e.g., US, Germany)?

---
**End of Notebook — Phase 4 Exploratory Data Analysis Complete.**
